# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** c23_design_space_optimizer     
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-19
### Design space optimization script
**Goal:** Optimize the structure by iterating over structural performance, circular performance, and total cost.
**Inputs:**
*   Search space definition
*   Trained surrogate model
*   Material input dataset

**Outputs:**
*   Optimized design candidates
*   Ranked solution set

# INPUT

In [ ]:
import importlib
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import config
import c23_stage_geometry as stage_geometry
import c24_stage_fitness as stage_fitness
import c25_stage_structural as stage_structural
import c26_stage_cost_matrix as stage_cost
import c27_stage_milp as stage_milp

importlib.reload(stage_geometry)
importlib.reload(stage_structural)
importlib.reload(stage_cost)
importlib.reload(stage_milp)
importlib.reload(stage_fitness)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# One-time: search space + stock dataset
json_path = config.DATA_IO_PATH / "search_space.json"
with open(json_path, "r", encoding="utf-8") as f:
    optimizer_search_space = json.load(f)

testing = False
if testing:
    file_path = config.TIMBER_STOCK_PATH / "complete_timber_small.csv"
else:
    file_path = config.TIMBER_STOCK_PATH / "complete_timber.csv"

read_attempts = [
    {"sep": ",", "encoding": "utf-8"},
    {"sep": ";", "encoding": "utf-8"},
    {"sep": ",", "encoding": "latin1"},
    {"sep": ";", "encoding": "latin1"},
]

df_input_stock = None
for opts in read_attempts:
    try:
        df_try = pd.read_csv(file_path, **opts)
        if df_try.shape[1] > 1:
            df_input_stock = df_try
            print(f"Loaded stock with sep='{opts['sep']}' and encoding='{opts['encoding']}'")
            break
    except Exception:
        pass

if df_input_stock is None:
    raise ValueError("Could not parse stock CSV with tested delimiter/encoding combinations.")

df_input_stock.columns = df_input_stock.columns.str.strip()

# One-time: surrogate bundle cache
MODEL_PREFIX = None
SURROGATE_BUNDLE, SURROGATE_BUNDLE_ERROR = stage_structural.prepare_surrogate_bundle(
    model_prefix=MODEL_PREFIX,
 )

if SURROGATE_BUNDLE is not None:
    print("Surrogate bundle loaded and cached for GA session.")
else:
    print("Surrogate bundle unavailable; GA uses synthetic force fallback.")
    print(f"Load reason: {SURROGATE_BUNDLE_ERROR}")

# GA configuration (balanced defaults)
GA_CONFIG = {
    "population_size": 30,
    "generations": 30,
    "elite_count": 2,
    "tournament_k": 3,
    "crossover_rate": 0.85,
    "mutation_rate": 0.25,
    "mutation_scale": 0.10,
    "utilization_threshold": 1.25,
    "weight_strategy": "cost-dominant",
    "penalty_fitness": 1e6,
    "max_retries_per_individual": 2,
}

# Fixed normalization constants for whole GA run
FIXED_NORMALIZATION_CONSTANTS = stage_fitness.get_default_normalization_constants()

print(f"Search space variables: {len(optimizer_search_space)}")
print(f"Stock elements: {len(df_input_stock)}")
print(f"Fixed normalization constants: {FIXED_NORMALIZATION_CONSTANTS}")
display(df_input_stock.head(3))

# OPTIMIZER

In [ ]:
# Custom GA operators (mixed discrete/continuous parameter dicts)
def _clip_continuous(value, bounds):
    return float(np.clip(value, float(bounds["min"]), float(bounds["max"])))


def _mutate_param(key, value, rules, mutation_scale=0.1):
    if rules["type"] == "continuous":
        span = float(rules["max"]) - float(rules["min"])
        sigma = max(span * float(mutation_scale), 1e-9)
        mutated = float(value) + float(np.random.normal(0.0, sigma))
        return _clip_continuous(mutated, rules)

    options = list(rules["options"])
    if len(options) <= 1:
        return value
    candidates = [opt for opt in options if opt != value]
    return random.choice(candidates)


def random_individual(search_space):
    return stage_geometry.sample_random_design(search_space)


def tournament_selection(population, fitnesses, k=3):
    idx = random.sample(range(len(population)), k=min(int(k), len(population)))
    best_idx = min(idx, key=lambda i: fitnesses[i])
    return population[best_idx].copy()


def crossover(parent_a, parent_b, search_space):
    child = {}
    for key, rules in search_space.items():
        a = parent_a[key]
        b = parent_b[key]

        if rules["type"] == "continuous":
            alpha = random.random()
            value = alpha * float(a) + (1.0 - alpha) * float(b)
            child[key] = _clip_continuous(value, rules)
        else:
            child[key] = random.choice([a, b])
    return child


def mutate(individual, search_space, mutation_rate=0.2, mutation_scale=0.1):
    out = individual.copy()
    for key, rules in search_space.items():
        if random.random() < float(mutation_rate):
            out[key] = _mutate_param(
                key=key,
                value=out[key],
                rules=rules,
                mutation_scale=float(mutation_scale),
            )
    return out


def initialize_population(search_space, size):
    return [random_individual(search_space) for _ in range(int(size))]


print("GA operators ready.")
print(f"Population size (default): {GA_CONFIG['population_size']}")

## 24_ Geometry

In [ ]:
# Optional one-sample geometry sanity check before GA run
geometry_out = stage_geometry.run_random_geometry_stage(
    optimizer_search_space=optimizer_search_space,
    sample_id=0,
)

df_vertices_example = geometry_out["df_vertices"]
df_edges_example = geometry_out["df_edges"]
df_geometry_overview_example = geometry_out["df_geometry_overview"]

print(
    f"Example geometry: {len(df_vertices_example)} nodes, "
    f"{len(df_edges_example)} members"
 )
display(df_geometry_overview_example[["edge_id", "V1", "V2", "length_m"]].head(5))

## 24-27 Iteration

In [ ]:
def evaluate_design_candidate(
    design_params,
    df_stock,
    bundle,
    fixed_norm_constants,
    config_dict,
    model_prefix=None,
    sample_id=0,
    export_dir=None,
    export_prefix=None,
    verbose=False,
 ):
    """Evaluate one design through c24-c27 stages and return fitness + diagnostics."""
    result = {
        "design_params": design_params,
        "status": "UNKNOWN",
        "fitness": float(config_dict["penalty_fitness"]),
        "reason": None,
        "fitness_result": None,
        "milp_status": None,
        "total_cost": float("inf"),
        "reuse_rate": 0.0,
        "waste_total": float("inf"),
    }

    try:
        # c23 geometry from explicit design
        geo_out = stage_geometry.run_geometry_from_design(
            design_params=design_params,
            sample_id=int(sample_id),
        )
        df_vertices = geo_out["df_vertices"]
        df_edges = geo_out["df_edges"]

        # c25 structural (GNN + utilization)
        structural_out = stage_structural.run_structural_stage(
            df_input_stock=df_stock,
            df_vertices=df_vertices,
            bundle=bundle,
            model_prefix=model_prefix,
            use_synthetic_fallback=True,
            gnn_margin=1.10,
            swap_width_depth_req=True,
            export_slots_path=None,
        )

        # c26 cost matrix
        cost_out = stage_cost.run_cost_matrix_stage(
            df_slots=structural_out["df_slots"],
            df_input_stock=df_stock,
            df_utilization_matrix=structural_out["df_utilization_matrix"],
            utilization_threshold=float(config_dict["utilization_threshold"]),
            target_stock_ids=df_stock["Member_ID"].dropna().astype(str).tolist(),
            export_cost_matrix_path=None,
            include_threshold_sweep=False,
            export_slot_analysis=False,
            quiet=True,
        )

        # c27 MILP
        milp_out = stage_milp.run_milp_stage(
            cost_matrix=cost_out["cost_matrix"],
            enriched_stock=cost_out["enriched_stock"],
            df_slots=structural_out["df_slots"],
            reclaimed_marker="RS",
            new_marker="NS",
            solver_msg=False,
            raise_on_infeasible_slots=False,
        )

        result["milp_status"] = milp_out["status"]
        if milp_out["status"] != "Optimal":
            result["status"] = "PENALIZED"
            result["reason"] = f"MILP status: {milp_out['status']}"
            return result

        # c24 fitness
        export_json_path = None
        export_csv_path = None
        if export_dir is not None and export_prefix is not None:
            export_json_path = Path(export_dir) / f"{export_prefix}_fitness_result.json"
            export_csv_path = Path(export_dir) / f"{export_prefix}_fitness_result.csv"

        fitness_out = stage_fitness.run_fitness_stage(
            df_results=milp_out["df_results"],
            enriched_stock=cost_out["enriched_stock"],
            df_slots=structural_out["df_slots"],
            total_cost=milp_out["total_cost"],
            weight_strategy=str(config_dict["weight_strategy"]),
            normalization_constants=fixed_norm_constants,
            run_sanity_checks=False,
            print_breakdown=False,
            export_json_path=export_json_path,
            export_csv_path=export_csv_path,
        )

        fitness_result = fitness_out["fitness_result"]
        result.update(
            {
                "status": "OK",
                "fitness": float(fitness_result["fitness"]),
                "reason": None,
                "fitness_result": fitness_result,
                "total_cost": float(fitness_result["cost_raw"]),
                "reuse_rate": float(fitness_result["reuse_rate"]),
                "waste_total": float(fitness_result["waste_total"]),
                "df_vertices": df_vertices,
                "df_edges": df_edges,
                "df_results": milp_out["df_results"],
            }
        )

        if verbose:
            print(
                f"OK | fitness={result['fitness']:.4f} | "
                f"cost={result['total_cost']:.2f} | reuse={result['reuse_rate']:.1f}%"
            )
        return result

    except Exception as exc:
        result["status"] = "PENALIZED"
        result["reason"] = str(exc)
        return result


print("Single-candidate evaluator ready.")
test_design = random_individual(optimizer_search_space)
test_eval = evaluate_design_candidate(
    design_params=test_design,
    df_stock=df_input_stock,
    bundle=SURROGATE_BUNDLE,
    fixed_norm_constants=FIXED_NORMALIZATION_CONSTANTS,
    config_dict=GA_CONFIG,
    model_prefix=MODEL_PREFIX,
    verbose=True,
 )
print(f"Evaluator smoke test status: {test_eval['status']}")
if test_eval['reason'] is not None:
    print(f"Reason: {test_eval['reason']}")

In [ ]:
# Smoke-test override (fast run)
GA_CONFIG.update({
    "population_size": 6,
    "generations": 3,
    "elite_count": 1,
    "tournament_k": 2,
})
print("Smoke-test GA config active:")
print({k: GA_CONFIG[k] for k in ["population_size", "generations", "elite_count", "tournament_k"]})

# OUTPUT

In [ ]:
# Run custom GA loop (balanced defaults)
population_size = int(GA_CONFIG["population_size"])
generations = int(GA_CONFIG["generations"])
elite_count = int(GA_CONFIG["elite_count"])
tournament_k = int(GA_CONFIG["tournament_k"])
crossover_rate = float(GA_CONFIG["crossover_rate"])
mutation_rate = float(GA_CONFIG["mutation_rate"])
mutation_scale = float(GA_CONFIG["mutation_scale"])

population = initialize_population(optimizer_search_space, population_size)
ga_history_rows = []
best_overall = None

for gen in range(generations):
    evals = []
    fitnesses = []

    for ind_idx, individual in enumerate(population):
        ev = evaluate_design_candidate(
            design_params=individual,
            df_stock=df_input_stock,
            bundle=SURROGATE_BUNDLE,
            fixed_norm_constants=FIXED_NORMALIZATION_CONSTANTS,
            config_dict=GA_CONFIG,
            model_prefix=MODEL_PREFIX,
            sample_id=ind_idx,
            verbose=False,
        )
        evals.append(ev)
        fitnesses.append(float(ev["fitness"]))

        ga_history_rows.append(
            {
                "generation": gen,
                "individual": ind_idx,
                "fitness": float(ev["fitness"]),
                "status": ev["status"],
                "reason": ev["reason"],
                "milp_status": ev.get("milp_status"),
                "total_cost": float(ev.get("total_cost", float("inf"))),
                "reuse_rate": float(ev.get("reuse_rate", 0.0)),
                "waste_total": float(ev.get("waste_total", float("inf"))),
                "design_params_json": json.dumps(ev["design_params"]),
            }
        )

    order = np.argsort(fitnesses)
    ranked = [evals[i] for i in order]
    best_gen = ranked[0]

    if best_overall is None or float(best_gen["fitness"]) < float(best_overall["fitness"]):
        best_overall = best_gen

    finite_fit = [f for f in fitnesses if np.isfinite(f)]
    mean_fit = float(np.mean(finite_fit)) if finite_fit else float("inf")
    feasible_count = int(sum(1 for r in evals if r["status"] == "OK"))
    print(
        f"Gen {gen + 1:02d}/{generations} | "
        f"best={float(best_gen['fitness']):.4f} | "
        f"mean={mean_fit:.4f} | "
        f"feasible={feasible_count}/{len(evals)}"
    )

    # Elitism + reproduction
    next_population = [ranked[i]["design_params"].copy() for i in range(min(elite_count, len(ranked)))]

    while len(next_population) < population_size:
        p1 = tournament_selection(population, fitnesses, k=tournament_k)
        p2 = tournament_selection(population, fitnesses, k=tournament_k)

        if random.random() < crossover_rate:
            child = crossover(p1, p2, optimizer_search_space)
        else:
            child = p1.copy()

        child = mutate(
            child,
            optimizer_search_space,
            mutation_rate=mutation_rate,
            mutation_scale=mutation_scale,
        )
        next_population.append(child)

    population = next_population

ga_history_df = pd.DataFrame(ga_history_rows)
ga_ranked_df = ga_history_df.sort_values(by=["fitness", "generation", "individual"]).reset_index(drop=True)

print("\nGA finished.")
print(f"Best fitness: {float(best_overall['fitness']):.4f}")
display(ga_ranked_df.head(10))

# EXPORT

In [ ]:
# Export GA history and best candidate payload
ga_history_path = config.GA_DATA_PATH / "c23_ga_history.csv"
ga_ranked_path = config.GA_DATA_PATH / "c23_ga_ranked.csv"
ga_best_design_path = config.GA_DATA_PATH / "c23_ga_best_design.json"

ga_history_df.to_csv(ga_history_path, index=False)
ga_ranked_df.to_csv(ga_ranked_path, index=False)

best_payload = {
    "fitness": float(best_overall["fitness"]),
    "status": best_overall["status"],
    "milp_status": best_overall.get("milp_status"),
    "total_cost": float(best_overall.get("total_cost", float("inf"))),
    "reuse_rate": float(best_overall.get("reuse_rate", 0.0)),
    "waste_total": float(best_overall.get("waste_total", float("inf"))),
    "design_params": best_overall["design_params"],
    "normalization_constants": FIXED_NORMALIZATION_CONSTANTS,
    "ga_config": GA_CONFIG,
}

with open(ga_best_design_path, "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)

print(f"GA history exported: {ga_history_path}")
print(f"GA ranked table exported: {ga_ranked_path}")
print(f"Best design exported: {ga_best_design_path}")

display(pd.DataFrame([best_payload]).drop(columns=["design_params", "ga_config"]))